# Letter-to-Word Dataset Generation

This notebook generates a synthetic dataset for training the letter-to-word prediction model. It loads common English words, applies various letter-level corruptions (noise) to simulate errors, and creates input-output pairs for the neural network.

In [ ]:
# Load word frequency data
import wordfreq
import pandas as pd

# Get the top 10,000 most common English words
words = wordfreq.top_n_list("en", n=10000)

# Filter words: keep only alphabetic words with length between 3 and 8
filtered_words = [
    w.upper()
    for w in words
    if w.isalpha() and 3 <= len(w) <= 8
]

# Display dataset size and sample words
print(len(filtered_words))  # ~3000–4000 words
print(filtered_words[:10])

7435
['THE', 'AND', 'FOR', 'THAT', 'YOU', 'WITH', 'THIS', 'WAS', 'ARE', 'HAVE']


## 1. Load and Filter English Words

Load the top 10,000 most common English words and filter them to include only alphabetic words with 3-8 characters.

In [ ]:
# Import random for error generation
import random

ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

def add_noise(word):
    """Add random letter-level errors to a word
    
    Supported error types:
    - substitute: replace a letter with a random one
    - delete: remove a letter
    - repeat: duplicate a letter
    - swap: swap two adjacent letters
    """
    word = list(word)
    # Randomly choose error type
    error_type = random.choice([
        "substitute",
        "delete",
        "repeat",
        "swap"
    ])

    # Select random position
    i = random.randint(0, len(word) - 1)

    # Apply the chosen error
    if error_type == "substitute":
        # Replace letter with random character
        word[i] = random.choice(ALPHABET)

    elif error_type == "delete" and len(word) > 3:
        # Remove letter (keep minimum length of 3)
        word.pop(i)

    elif error_type == "repeat":
        # Duplicate letter at position
        word.insert(i, word[i])

    elif error_type == "swap" and i < len(word) - 1:
        # Swap with adjacent letter
        word[i], word[i+1] = word[i+1], word[i]

    return word

## 2. Define Noise Functions

Create a function to add realistic letter-level errors to words, simulating common OCR and typing mistakes.

In [ ]:
# Import regex for letter extraction
import re

# Build processed dataset with input letters as lists and outputs as letter lists, then save.

# Use existing dataset_df if present, otherwise construct from rows
if 'dataset_df' in globals() and isinstance(dataset_df, pd.DataFrame):
    df = dataset_df.copy()
else:
    df = pd.DataFrame(rows)

def to_letter_list(x):
    """Convert input to a list of individual uppercase letters"""
    if isinstance(x, list):
        return [str(ch).upper() for ch in x]
    # Extract only alphabetic characters from string
    s = str(x)
    letters = re.findall(r"[A-Za-z]", s)
    return [ch.upper() for ch in letters]

# Ensure input_letters column contains list of single-character strings
df['input_letters'] = df['input_letters'].apply(to_letter_list)

# Create target letter list from the target word column
df['output_real_letters'] = df['target'].astype(str).apply(lambda s: [c.upper() for c in s])

# Save processed dataset in multiple formats for flexibility
df.to_csv('processed_dataset.csv', index=False)  # CSV format
df.to_pickle('processed_dataset.pkl')            # Pickle format (preserves data types)

# Expose processed dataframe for further use
dataset_processed = df